## Data Source and Scope

### Data Source

The data used in this project comes from the **Encuesta Nacional de Ocupación y Empleo (ENOE)** 
(National Survey of Occupation and Employment), produced by Mexico's **Instituto Nacional de 
Estadística y Geografía (INEGI)**.

ENOE provides detailed information about the demographic, educational and labour-market 
characteristics of the Mexican population. The survey is particularly relevant to this project 
because it contains person-level information that can be used to analyse women's participation 
in the labour market and the socioeconomic factors associated with employment outcomes.

The original ENOE data is distributed across several related datasets. For this project, the 
main sources are:

- **SDEM:** Sociodemographic characteristics.
- **COE1:** Employment and occupation questionnaire – Part I.
- **COE2:** Employment and occupation questionnaire – Part II.

These datasets are integrated using common household- and person-level identifiers to create 
the analytical dataset used throughout the project.

### Why 2024?

The **2024 ENOE** was selected as the main source of microdata because it provides relatively 
recent labour-market information while offering all four quarterly datasets required to build 
a full-year analytical dataset.

Using the four quarters of the same year provides a sufficiently large sample for exploratory 
data analysis, statistical analysis, machine-learning models and segmentation, while keeping 
the project computationally manageable.

The project initially considered using historical census microdata from IPUMS International. 
However, access to those data requires account approval. Given the project's delivery timeline, 
ENOE 2024 was selected as the primary microdata source because the official INEGI datasets are 
available for direct download.

The project therefore focuses on **women's labour-market outcomes in Mexico during 2024**, 
while additional international and historical indicators can be incorporated separately for 
comparative and time-series analyses.

## 1. Data Loading

The ENOE 2024 data is distributed across multiple datasets containing different types of information.

For the first quarter of 2024, three datasets were loaded:

- **SDEM:** Sociodemographic characteristics of individuals.
- **COE1:** Employment and occupation questionnaire – Part I.
- **COE2:** Employment and occupation questionnaire – Part II.

The SDEM dataset will be used as the main person-level dataset, while COE1 and COE2 will provide additional labour-market information.

In [2]:
import pandas as pd

# Load the datasets for the first quarter of 2024
df_sdem_1 = pd.read_csv("enoe_2024_trim1_csv/ENOE_SDEMT124.csv",
    encoding="latin1")
df_coe1_1 = pd.read_csv("enoe_2024_trim1_csv/ENOE_COE1T124.csv",
    encoding="latin1")
df_coe2_1 = pd.read_csv("enoe_2024_trim1_csv/ENOE_COE2T124.csv",
    encoding="latin1")

## 2. Initial Data Validation

Before merging the datasets, their dimensions were reviewed to understand the number of observations and variables available in each source.

SDEM contains more observations than COE1 and COE2 because it includes sociodemographic records for a broader population, while the employment questionnaires apply only to eligible respondents.

In [3]:
#Pritn the shape of each dataset to verify they were loaded correctly
print("SDEM:", df_sdem_1.shape)
print("COE1:", df_coe1_1.shape)
print("COE2:", df_coe2_1.shape)

SDEM: (423866, 114)
COE1: (347366, 189)
COE2: (347366, 137)


In [4]:
df_sdem_1.head()

,r_def,loc,mun,est,est_d_tri,est_d_men,ageb,t_loc_tri,t_loc_men,cd_a,...,scian,t_tra,emp_ppal,tue_ppal,trans_ppal,mh_fil2,mh_col,sec_ins,tipo,mes_cal
0,0,,10,20,123,109,0,1,1,1,...,7,1,1,2,0,3,1,4,2,2
1,0,,9,20,126,117,0,2,2,1,...,0,1,0,0,0,0,0,0,2,1
2,0,,13,20,123,,0,1,,1,...,4,1,1,1,0,1,1,8,1,96
3,0,,13,20,123,,0,1,,1,...,0,0,0,0,0,0,0,0,1,96
4,0,,5,20,123,,0,1,,1,...,14,1,2,2,0,3,2,2,1,96


## 3. Identification of the Merge Key

Before merging the datasets, a set of variables was evaluated as a potential composite key for identifying individuals across SDEM, COE1 and COE2.

The ENOE datasets do not rely on a single identifier to uniquely identify each person. Therefore, a combination of household- and person-level variables was initially evaluated as a composite key.

The initial candidate key was:

`cd_a + ent + con + v_sel + n_hog + h_mud + n_ren`

| Feature | Description according to ENOE data dictionary|
|:--------|:------------|
| `cd_a`  | Self-representing city |
| `ent`   | State (federal entity) |
| `con`   | Control number |
| `v_sel` | Selected dwelling |
| `n_hog` | Household number |
| `h_mud` | Moved household indicator |
| `n_ren` | Person's row number within the household |

Instead of performing the merge directly, the uniqueness of this composite key was tested in each dataset to prevent unintended many-to-many relationships.

In [5]:
#Now we are checking if our features for the merge is all our datasets, we will check if the columns are present in each dataset
keys = ['cd_a', 'ent', 'con', 'v_sel', 'n_hog', 'h_mud', 'n_ren']

for name, df in {
    'SDEM': df_sdem_1,
    'COE1': df_coe1_1,
    'COE2': df_coe2_1
}.items():
    print(f"\n{name}:")
    print([col for col in keys if col in df.columns])


SDEM:
['cd_a', 'ent', 'con', 'v_sel', 'n_hog', 'h_mud', 'n_ren']

COE1:
['cd_a', 'ent', 'con', 'v_sel', 'n_hog', 'h_mud', 'n_ren']

COE2:
['cd_a', 'ent', 'con', 'v_sel', 'n_hog', 'h_mud', 'n_ren']


In [ ]:
#Before merging, we will check the number of duplicates in each dataset by keys
print(
    "Duplicates SDEM:",
    df_sdem_1.duplicated(subset=keys).sum())

print(
    "Duplicates COE1:",
    df_coe1_1.duplicated(subset=keys).sum())

print(
    "Duplicates COE2:",
    df_coe2_1.duplicated(subset=keys).sum())

Duplicates SDEM: 23574
Duplicates COE1: 18834
Duplicates COE2: 18834


In [7]:
#First check some of the duplicates in SDEM
duplicates_sdem = (df_sdem_1[df_sdem_1.duplicated(subset=keys, keep=False)].sort_values(keys))
duplicates_sdem.head(20)

,r_def,loc,mun,est,est_d_tri,est_d_men,ageb,t_loc_tri,t_loc_men,cd_a,...,scian,t_tra,emp_ppal,tue_ppal,trans_ppal,mh_fil2,mh_col,sec_ins,tipo,mes_cal
102060,0,,6,20,123,,0,1,,1,...,16,1,2,2,0,3,2,7,1,96
318221,0,,6,20,123,109,0,1,1,1,...,12,1,1,2,0,3,1,4,2,3
75542,0,,6,20,123,,0,1,,1,...,16,1,2,2,0,3,2,7,1,96
357846,0,,6,20,123,109,0,1,1,1,...,0,1,0,0,0,0,0,0,2,3
62193,0,,6,20,123,,0,1,,1,...,0,0,0,0,0,0,0,0,1,96
318222,0,,6,20,123,109,0,1,1,1,...,0,1,0,0,0,0,0,0,2,3
227973,0,,6,20,123,109,0,1,1,1,...,8,1,1,1,0,1,7,8,2,3
274265,0,,6,20,123,,0,1,,1,...,0,1,0,0,0,0,0,0,1,96
3045,0,,15,20,123,,0,1,,1,...,0,1,0,0,0,0,0,0,1,96
304987,0,,15,20,123,114,0,1,1,1,...,8,1,2,2,0,3,2,2,2,1


In [8]:
duplicates_sdem[keys + ['n_ent', 'per', 'tipo', 'mes_cal']].head(20)

,cd_a,ent,con,v_sel,n_hog,h_mud,n_ren,n_ent,per,tipo,mes_cal
102060,1,9,40003,1,1,0,1,1,124,1,96
318221,1,9,40003,1,1,0,1,1,124,2,3
75542,1,9,40003,1,1,0,2,1,124,1,96
357846,1,9,40003,1,1,0,2,1,124,2,3
62193,1,9,40003,1,1,0,3,1,124,1,96
318222,1,9,40003,1,1,0,3,1,124,2,3
227973,1,9,40003,4,1,0,1,1,124,2,3
274265,1,9,40003,4,1,0,1,1,124,1,96
3045,1,9,40026,1,1,0,1,4,124,1,96
304987,1,9,40026,1,1,0,1,4,124,2,1


In [9]:
keys_base = ['cd_a', 'ent', 'con', 'v_sel', 'n_hog', 'h_mud', 'n_ren']

candidates = [
    keys_base,
    keys_base + ['n_ent'],
    keys_base + ['tipo'],
    keys_base + ['n_ent', 'tipo']
]

for keys_test in candidates:
    print("\nKeys:", keys_test)

    for name, df in {
        'SDEM': df_sdem_1,
        'COE1': df_coe1_1,
        'COE2': df_coe2_1
    }.items():
        print(
            name,
            df.duplicated(subset=keys_test).sum()
        )


Keys: ['cd_a', 'ent', 'con', 'v_sel', 'n_hog', 'h_mud', 'n_ren']
SDEM 23574
COE1 18834
COE2 18834

Keys: ['cd_a', 'ent', 'con', 'v_sel', 'n_hog', 'h_mud', 'n_ren', 'n_ent']
SDEM 18944
COE1 15133
COE2 15133

Keys: ['cd_a', 'ent', 'con', 'v_sel', 'n_hog', 'h_mud', 'n_ren', 'tipo']
SDEM 0
COE1 0
COE2 0

Keys: ['cd_a', 'ent', 'con', 'v_sel', 'n_hog', 'h_mud', 'n_ren', 'n_ent', 'tipo']
SDEM 0
COE1 0
COE2 0
